# Table extraction — where do the seconds go?

Pick **any form and any PDF from production**, run that form's table field with full timing
instrumentation, and get a per-operation breakdown plus a waterfall showing what actually
overlapped.

**Kernel: the `topics` conda env** (`/home/ubuntu/miniconda3/envs/topics/bin/python`). The
backend's DSPy/litellm stack lives there, not in `base`.

### What gets measured

| Operation | What it is | Calls |
|---|---|---|
| `record_discovery` | which records the paper reports — key columns only | 1 |
| `recall_audit` | second pass: *what did discovery miss?* | 1 |
| `slot_fill_set` | fill many records in one call (the current default) | ceil(n/k) |
| `slot_fill_row` | fill one record's attributes (`EXTRACTION_BATCH_VALUES=0`, and the fallback when a set call fails twice) | 1 per record |
| `refill` | targeted re-extraction of records that came back empty | 0–2 rounds |

Each LLM call is timed two ways: **our wall clock** around the awaited call, and litellm's own
`_response_ms` (`api_s`). `wall − api` is our own overhead — adapter rendering, JSON parsing,
reconciliation. The difference between `api_s_sum` and `busy_s` is the concurrency: sum adds
every call, `busy_s` is the union of their intervals, so `sum / busy` is the real speedup the
`asyncio.gather` fan-outs are buying.

### Two sources, both here

1. **Live run** (sections 1–4) — real Bedrock calls on a paper you choose. Costs tokens.
2. **Past production jobs** (section 5) — `llm_history.metadata.duration_ms` and the `jobs`
   table. Free, and it includes the Celery queue wait that an in-process run can't see.

DSPy's response cache is **disabled** by `bootstrap()`. A cached response returns in
microseconds, so timing one is not timing the pipeline.

## 0 · Parameters

Leave `FORM_ID` / `FIELD` / `DOC_IDS` as `None` on the first pass — section 1 prints the prod
tables to choose from, and the auto-pick fills them in so the notebook runs top-to-bottom
without editing anything.

In [ ]:
# Nothing here needs to be known in advance. Leave it all as-is on the first run:
# section 1 prints every table field on prod, section 2 prints that project's PDFs.
# FORM_ID / DOC_IDS accept names as well as ids. Strings must be quoted.
#
# ── what to profile ────────────────────────────────────────────────────────
PROJECT_NAME = "Analgesics"
FORM_ID      = "Acute Dental Pain — Dichotomous Outcomes"
FIELD        = "dichotomous_outcomes"   # table field on that form, e.g. "outcomes". None → its only/first one
DOC_IDS         = ["A single-tablet fixed-dose combination of racemic ibuprofen/paracetamol in the management of moderate to severe postoperative dental pain in adult and adolescent patients: a multicenter, two-stage, randomized, double-blind, parallel-group, placebo-control"]    # ids and/or title fragments: ["ketoprofen", "meloxicam"]. None → first N_PAPERS
N_PAPERS        = 1       # used only when DOC_IDS is None. Each paper is a full extraction — the cost dial

# ── how to run it ──────────────────────────────────────────────────────────
MODEL_OVERRIDE   = None   # e.g. "anthropic/claude-haiku-4-5" — None = the form's production model
USE_MODEL_ROUTER = True   # True mirrors production (ModelRouter + dspy.context + circuit breaker)
PARALLEL_PAPERS  = False  # False = papers in sequence, so per-step numbers are uncontended

# ── prod history section ───────────────────────────────────────────────────
PROD_LOOKBACK_DAYS = 30
PROD_JOB_ID        = None  # pin one job instead of the whole window

SAVE = True               # write CSVs + JSON under eval/profiling/outputs/

In [ ]:
import sys, asyncio, pandas as pd
sys.path.insert(0, "/home/ubuntu/evistream/backend/eval/profiling")
import table_timing as tt

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

RUNTIME = tt.bootstrap()          # sys.path + prod secrets + DSPy config, cache off
pd.Series(RUNTIME).to_frame("value")

Read `slot_filling` in that table before anything else — it decides the shape of the whole run.
`EXTRACTION_BATCH_VALUES` is unset in production and **defaults to `1`** in
`runtime_builders`, i.e. set-at-a-time: one call carries up to ~40 records. Export
`EXTRACTION_BATCH_VALUES=0` before starting the kernel to measure the row-at-a-time path
instead (which is also what the set path falls back to when a batch fails twice).

`EXTRACTION_PROMPT_CACHE=1` means the paper is sent once and re-read from Anthropic's cache —
the first call of a paper pays the write, every later call reads it, which is exactly what the
`cache_read` / `cache_write` columns show.

## 1 · Pick a form and table field from production

`runs_as` is the column that matters: it mirrors the activation test in
`runtime_builders.build_schema_classes`, so a field that says `single_pass` here runs as one
call in production no matter what the form UI shows.

In [ ]:
forms = tt.list_table_forms()

# The valid values for PROJECT_NAME, so it can be copied instead of guessed.
print("PROJECT_NAME options:")
display(forms.groupby("project")
             .agg(forms=("form_id", "nunique"), table_fields=("field", "size"),
                  keyed=("keyed_pipeline", "sum"))
             .sort_values("table_fields", ascending=False))

forms = tt.match_project(forms, PROJECT_NAME)   # exact name wins over substring

print(f"\n{len(forms)} table field(s) · keyed: {int(forms.keyed_pipeline.sum())}"
      f"  — copy a form_id (or the form_name) into FORM_ID above to pin one")
forms[["project", "form_name", "field", "runs_as", "n_key", "n_attr",
       "key_columns", "form_id"]].head(40)

In [ ]:
# FORM_ID takes a UUID *or* a form name; FIELD narrows within it. With both None,
# the first keyed field is picked. Ambiguous matches raise and print the candidates.
sel, notes = tt.resolve_selection(forms, form=FORM_ID, field=FIELD)
for n in notes:
    print("⚠", n)

FORM_ID, FIELD = sel.form_id, sel.field
schema_def = tt.load_schema_def(FORM_ID)
sel.to_frame("selected")

## 2 · Pick the paper(s)

Any document in the form's project that has markdown in S3 — that's the set the pipeline can
actually run on, so the table below is the menu.

`DOC_IDS` takes whatever you have to hand: document ids, filename/title fragments, or a mix —
one string or a list. `["ketoprofen", "meloxicam"]` is a valid selection. Left as `None`, the
first `N_PAPERS` rows run, so the notebook works untouched.

Cost scales linearly with the number of papers; wall clock does too unless you set
`PARALLEL_PAPERS = True`. Bigger papers and wider tables are where the interesting time goes.

In [ ]:
docs = tt.list_documents(sel.project_id, limit=60)
print(f"{len(docs)} document(s) with markdown in project {sel.project!r}")
docs.head(30)

In [ ]:
# DOC_IDS accepts document ids, filename/title fragments, or a mix — one string or a
# list. None → the first N_PAPERS rows above. A fragment matching 2 papers raises.
doc_ids, picked = tt.resolve_documents(docs, DOC_IDS, n_default=N_PAPERS)
print(f"running {len(doc_ids)} paper(s):")
for i, (d, lbl) in enumerate(zip(doc_ids, picked), 1):
    print(f"  {i}. {lbl[:70]}  [{d}]")

papers = tt.fetch_papers(doc_ids)     # S3 → temp files, same bytes production sends
pd.DataFrame([{k: p.get(k) for k in ("label", "page_count", "chars", "est_tokens", "doc_id")}
              for p in papers])

## 3 · Run it, instrumented

This makes real LLM calls. Cost per paper is `2 + ceil(records/40)` calls on the set-at-a-time
default, or `2 + records` on the row-at-a-time path, plus any refill round.

`profile_field` builds the extractor through `build_schema_classes` — the same selection
production makes — and runs it inside `ModelRouter.run_with_routing`, so the LM, the
`CachingChatAdapter`, and the circuit breaker all behave as they do in a real job.

In [ ]:
run = await tt.profile_field(
    schema_def, FIELD, papers,
    use_model_router=USE_MODEL_ROUTER,
    model_override=MODEL_OVERRIDE,
    parallel_papers=PARALLEL_PAPERS,
)
print(f"pipeline: {run.runs_as}  ·  signature: {run.sig_class}  ·  model: {run.model}")
if run.errors:
    print("ERRORS:", run.errors)
print() 
print(tt.overhead_report(run))

## 4 · Where the time went

### 4a · Per operation

`busy_s` = wall-clock contribution (union of intervals). `api_s_sum` = every call's latency
added up. `api_s_sum / busy_s` is the concurrency actually achieved — 1.0 means the step ran
strictly one call at a time and every second of it is on the critical path.

In [ ]:
steps = tt.step_summary(run)
steps

### 4b · The operation structure

Spans, nested. `meta_n` is how many records that span was responsible for — it's how you tell a
batched fill from the row-at-a-time fallback, and how you spot a refill round doing real work.
A `refill` span of ~0.000s means every frozen record came back filled on the first pass.

In [ ]:
tt.phase_summary(run)

### 4c · Per paper

`non_llm_s` is our own Python — parsing, enum normalisation, record reconciliation, the quote
checks in the recall-audit gate. If it is more than a second or two, the bottleneck is not the
model.

In [ ]:
tt.paper_summary(run)

### 4d · Every call

`step` comes from the extractor's own sub-module (`record_discovery`, `recall_auditor`,
`set_slot_filler`, `row_slot_filler`) — stamped at construction, never inferred from ordering.
`step_from_prompt` is the independent re-derivation from the rendered prompt by
`utils/llm_call_labels.classify_step`, i.e. the exact code that labels production cost rows.

The two agree except for record discovery, which classifies as `extract` from the prompt alone
(its prompt carries none of the three keyed marker fields — production tells it apart only by
comparing every call of the run). The next cell checks that rule instead of raw equality.

`error == "superseded_by_retry"` marks a call whose output failed to parse and was re-asked
(DSPy's JSONAdapter fallback); `shape == "json"` is the tell. Those seconds are pure waste and
worth chasing.

In [ ]:
calls = run.profiler.calls_df()
cols = ["seq", "paper", "step", "step_from_prompt", "span_name", "start_s", "wall_s", "api_s",
        "overhead_s", "prompt_tokens", "completion_tokens", "cache_read", "cache_write",
        "cost", "shape", "finish_reason", "truncated", "error"]
calls[[c for c in cols if c in calls.columns]]

In [ ]:
mismatch = tt.label_check(run)
print("label mismatches vs production telemetry:", len(mismatch))
mismatch if len(mismatch) else "all labels agree with utils/llm_call_labels"

### 4e · Waterfall

Grey bars are operations, coloured bars are LLM calls. Overlap = a `gather` fan-out; a staircase
means the latency lands on the critical path in full. The first call of a fan-out is deliberately
alone — Anthropic only populates a cache entry once a response has begun, so firing every branch
at once would make them all pay the 1.25× cache-write rate.

In [ ]:
for p in sorted({c.paper for c in run.profiler.calls}):
    display(tt.waterfall(run, paper=p))

In [ ]:
if SAVE:
    out_dir = tt.save_outputs(run, tag=f"{FIELD}")
    print("saved →", out_dir)

## 5 · What production actually took (free — no LLM calls)

The live run above is one paper under ideal conditions. These two tables are the real thing:
recent jobs with their **queue wait**, and per-step latency across every call in the window.

`queue_s` is `started_at − created_at` — the Celery wait. It is invisible to an in-process
profiler and is often the larger half of "why did this job take so long".

In [ ]:
jobs = tt.prod_jobs(project_id=sel.project_id, limit=25)
jobs

In [ ]:
prod = tt.prod_step_summary(
    schema_name=sel.schema_name,
    job_id=PROD_JOB_ID,
    days=PROD_LOOKBACK_DAYS,
)
print(f"schema {sel.schema_name} · last {PROD_LOOKBACK_DAYS} days")
prod

`with_timing` < `calls` means some rows predate the Aug 2026 per-call labels: their step is
recovered by re-classifying the stored prompt, but they carry no duration, so they count toward
volume and cost, not latency. If it is 0, widen the window or pick a schema that has run since.

Two step names to read carefully here:

* `extract` rows are the form's **scalar** signatures, not table work — the schema runs those in
  the same job. Only `record_discovery` / `recall_audit` / `slot_fill` / `slot_fill_row` /
  `refill` belong to the table field.
* production stores `slot_fill` where this profiler says `slot_fill_set`; the comparison below
  renames via `tt.PROD_STEP_NAME` so the two line up.

The `cache_write` row is the one paying the 1.25× write rate — normally record discovery, since
it is the first call to touch the paper.

In [ ]:
# Live run vs production, per step.
if len(prod) and len(steps):
    live = steps.assign(step=steps.step.map(lambda s: tt.PROD_STEP_NAME.get(s, s)))
    cmp = (live.groupby("step", as_index=False)
               .agg(calls_live=("calls", "sum"), api_s_median_live=("api_s_median", "median"))
               .merge(prod[["step", "calls", "median_s", "p95_s"]].rename(
                   columns={"calls": "calls_prod", "median_s": "median_s_prod",
                            "p95_s": "p95_s_prod"}),
                   on="step", how="outer"))
    display(cmp)
else:
    print("nothing to compare — no prod rows in the window, or the live run made no calls")

## Notes — what changes these numbers

* **`EXTRACTION_BATCH_VALUES`** — unset defaults to `1`: set-at-a-time, up to ~40 records per
  call, so a 9-row table costs 3 calls total. Set it to `0` before starting the kernel for the
  row-at-a-time path (2 + one call per record), which is also the automatic fallback when a set
  call fails twice. Set-at-a-time was once disabled after a cost regression; the env default has
  since flipped back, so always read `slot_filling` in the runtime table rather than assuming.
* **`EXTRACTION_PROMPT_CACHE`** — with it off, every call re-sends the whole paper: latency and
  cost both climb, visible as `cache_read == 0` on every row.
* **DSPy's response cache** — `bootstrap()` disables it. Re-enable only if you *want* to measure
  cache hits (`bootstrap(disable_dspy_cache=False)`).
* **`MODEL_OVERRIDE`** — separates model latency from pipeline structure. The step *counts* stay
  the same; only `api_s` moves.
* **`USE_MODEL_ROUTER=False`** — skips the circuit breaker and `dspy.context`. Slightly faster,
  no longer production-shaped.
* **Agentic fields** — a field whose strategy resolves to `agentic` runs the Claude Agent SDK
  extractor, whose internal turns are not DSPy forwards, so only the outer span is timed.
* **Retries above the extractor** — `StagedPipeline` re-runs a whole extractor when every field
  comes back empty. This notebook profiles one extractor call, so that outer retry shows up in
  production numbers (section 5) but not here.
* **The `topics` kernel** matters: in `base`, `import dspy` fails or resolves to a different
  version than production runs.